In [1]:
import fitz  # PyMuPDF

doc = fitz.open("../data/books/WithEachAndEveryBreath_210603.pdf")
full_text = ""

for page in doc:
    full_text += page.get_text() + "\n"

with open("breath_text.txt", "w", encoding="utf-8") as f:
    f.write(full_text)

In [2]:
full_text

"\n1\nWith Each & Every Breath\nA Guide To Meditation\n\ue04canissaro Bhikkhu\n(Geoﬀrey DeGraﬀ)\n\n2\nCopyright 2013 Ṭhānissaro Bhikkhu\n\ue004is work is licensed under the Creative Commons\nAttribution-NonCommercial 4.0 Unported. To see a\ncopy of this license visit\nhttp://creativecommons.org/licenses/by-nc/4.0/.\n“Commercial” shall mean any sale, whether for\ncommercial or non-pro\x11t purposes or entities.\nQuestions about this book may be addressed to\nMetta Forest Monastery\nValley Center, CA 92082-1409\nU.S.A.\nAdditional resources\nMore Dhamma talks, books and translations by\nṬhānissaro Bhikkhu are available to download in digital\naudio and various ebook formats at dhammatalks.org.\nPrinted copy\nA paperback copy of this book is available free of charge.\nTo request one, write to: Book Request, Metta Forest\nMonastery, PO Box 1409, Valley Center, CA 92082\nUSA.\n\n3\nIntroduction\nMEDITATION: WHAT & WHY\nMeditation is training for the mind, to help it develop the strengths\na

In [4]:
from pdfminer.high_level import extract_text

text = extract_text("../data/books/WithEachAndEveryBreath_210603.pdf")
with open("clean_text.txt", "w", encoding="utf-8") as f:
    f.write(text)


Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBB

In [5]:
text

"\x0cWith Each & Every Breath\n\nA Guide To Meditation\n\n\ue04canissaro Bhikkhu\n(Geoﬀrey DeGraﬀ)\n\n1\x0cCopy r i g ht   2 0 1 3   Ṭ hā ni s s a ro  Bhi k k hu\n\n\ue004is work is licensed under the Creative Commons\nAttribution-NonCommercial 4.0 Unported. To see a\ncopy of this license visit\nhttp://creativecommons.org/licenses/by-nc/4.0/.\n“Commercial” shall mean any sale, whether for\ncommercial or non-pro\x00t purposes or entities.\n\nQue st i ons   a b out   t hi s   b ook   may   b e   a ddr e s s e d  to\n\nMetta Forest Monastery\nValley Center, CA 92082-1409\nU.S.A.\n\nAddi t i ona l   r e s ourc e s\n\nMore Dhamma talks, books and translations by\nṬhānissaro Bhikkhu are available to download in digital\naudio and various ebook formats at dhammatalks.org.\n\nP r i nt e d  copy\n\nA paperback copy of this book is available free of charge.\nTo request one, write to: Book Request, Metta Forest\nMonastery, PO Box 1409, Valley Center, CA 92082\nUSA.\n\n2\x0cIntroduction\n\nMEDITAT

In [6]:
from pdf2image import convert_from_path
import pytesseract

pages = convert_from_path("../data/books/WithEachAndEveryBreath_210603.pdf", dpi=300)
full_text = ""

for page in pages:
    text = pytesseract.image_to_string(page, lang="eng")
    full_text += text + "\n"

with open("breath_ocr.txt", "w", encoding="utf-8") as f:
    f.write(full_text)


In [7]:
full_text

"ope\nKk\n\nfo“ ~-. & 4\nSee, *\nCW\n\n< ioe 3 , es\nee WITH\n} EACH\n\n\\\n\nG\nEVERY\nBREAIA\n\nA GUIDETO MEDITATION\n\n\nWith Each & Every Breath\n\nA GUIDE TO MEDITATION\n\nThanissaro Bhikkhu\n(Geoffrey DeGraff)\n\nCOPYRIGHT 2013 THANISSARO BHIKKHU\nThis work is licensed under the Creative Commons\nAttribution- NonCommercial 4.0 Unported. To see a\ncopy of this license visit\nhttp://creativecommons.org/licenses/by-nc/4.0/.\n“Commercial” shall mean any sale, whether for\ncommercial or non-profit purposes or entities.\n\nQUESTIONS ABOUT THIS BOOK MAY BE ADDRESSED TO\nMetta Forest Monastery\nValley Center, CA 92082-1409\nU.S.A.\n\nADDITIONAL RESOURCES\nMore Dhamma talks, books and translations by\nThanissaro Bhikkhu are available to download in digital\naudio and various ebook formats at dhammatalks.org.\n\nPRINTED COPY\n\nA paperback copy of this book is available free of charge.\n‘To request one, write to: Book Request, Metta Forest\nMonastery, PO Box 1409, Valley Center, CA 92082\n

In [19]:
import re

def clean_ocr_text(text: str) -> str:
    # Remove isolated numbers likely to be page numbers (surrounded by newlines)
    text = re.sub(r'\n+\s*(\d{1,3})\s*\n+', '\n\n', text)

    # Replace 3+ line breaks with double newlines (true paragraph break)
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Fix list item splits like "1)\ntext" → "1) text"
    text = re.sub(r'(\n)(\d+\))\s*\n', r'\n\2 ', text)

    # Fix mid-paragraph line breaks
    def fix_linebreaks_in_paragraphs(t: str) -> str:
        lines = t.split("\n")
        new_lines = []

        for i in range(len(lines)):
            if i > 0:
                prev = new_lines[-1].strip()
                curr = lines[i].strip()
                # Join if:
                # - Previous line doesn't end with sentence punctuation
                # - And current line doesn't start with capital letter or bullet
                if (prev and not re.search(r'[.!?:;]["\')\]]?$', prev)) and \
                   (curr and not re.match(r'^[A-Z•]', curr)):
                    new_lines[-1] += " " + curr
                    continue
            new_lines.append(lines[i])
        return "\n".join(new_lines)

    text = fix_linebreaks_in_paragraphs(text)

    return text

    # Load and clean
with open("breath_ocr.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

cleaned_text = clean_ocr_text(raw_text)

# Save result
with open("breath_cleaned.txt", "w", encoding="utf-8") as f:
    f.write(cleaned_text)


In [22]:
import re

def clean_ocr_text(text: str) -> str:
    # Remove isolated numbers (page numbers) between line breaks
    # Matches: \n<digits>\n — only if digits are alone
    text = re.sub(r'\n\d{1,3}\n', '\n', text)

    # Optional: Remove numbers with whitespace before/after (in case OCR added extra spaces)
    text = re.sub(r'\n\s*\d{1,3}\s*\n', '\n', text)

    # Remove repeated newlines (>2)
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Remove headers/footers if they appear — customize with known ones
    text = re.sub(r'(?i)with each and every breath', '', text)

    # Fix broken lines inside paragraphs:
    # If a line ends without punctuation and the next line doesn't start with a capital or bullet
    # This may need further refinement depending on the text
    #text = re.sub(r'(?<![\.\?!])\n(?=\w)', ' ', text)

    # Ensure clean paragraph spacing
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{2,}', '\n\n', text)

    return text.strip()

# Load and clean
with open("breath_ocr.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

cleaned_text = clean_ocr_text(raw_text)

# Save result
with open("breath_cleaned.txt", "w", encoding="utf-8") as f:
    f.write(cleaned_text)